In [1]:
from parameter.aparameter import AParameter
from strategy.strategy_factory import StrategyFactory
from transformer.transformer import Transformer
from backtester.backtester import Backtester
from processor.server_processor import ServerProcessor
from datetime import datetime, timedelta
from strategy.strategy import Strategy
import pandas as pd
from tqdm import tqdm
from database.adatabase import ADatabase
import matplotlib.pyplot as plt

In [2]:
db = ADatabase("sapling")

In [3]:
db.connect()
kpi = db.retrieve("kpi")
db.disconnect()

In [4]:
strategy_kpis = kpi.sort_values("return",ascending=False).groupby(["strategy"]).first().sort_values("return",ascending=False).reset_index()

In [5]:
strategy_kpis

,strategy,number_of_trades,standard_deviation,coefficient_of_variance,sharpe,return,holding_period,positions,stop_loss,ascending
0,STOCHASTIC_OSCILLATOR,520,318.7233,0.9389,3.0039,957.4090,5,5,0.05,False
1,RSI,520,264.1645,0.9018,3.1913,843.0273,5,5,0.05,True
2,BOLLINGER,520,277.1389,0.8969,3.0225,837.6532,5,5,0.05,False
3,COEFFICIENT_OF_VARIANCE,520,230.0475,0.8675,3.0079,691.9580,5,5,0.05,False
4,MACD,520,207.7345,0.8008,3.0077,624.8015,5,5,0.05,True


In [6]:
db = ADatabase("sapling")
market = ADatabase("market")
market.connect()
tickers = market.retrieve("russell1000")["ticker"].values
market.disconnect()
strategy_kpis["tickers"] = [tickers for i in range(strategy_kpis.index.size)]
query = strategy_kpis.to_dict("records")[0]

In [7]:
analysis = []
db.connect()
recs = []

try:
    start = datetime.now() - timedelta(days=365.25)
    end = datetime.now() - timedelta(hours=24)
    parameter = AParameter()
    parameter.build(query)
    strategy = StrategyFactory.build(parameter)
    simulation = Transformer.local_transform(strategy,start,end)
    trades = Backtester.backtest(strategy,simulation)
    portfolio = Backtester.portfolio(trades)
    recommendations = Backtester.recommendations(trades)
    kpi = Backtester.kpi(trades,portfolio)
    results = ServerProcessor.server_format(strategy,trades,portfolio,recommendations,kpi)
    recs.extend(results["recommendations"])
except Exception as e:
    print(str(e))
db.disconnect()

In [8]:
recommendations = pd.DataFrame(recs)

In [9]:
recommendations

,date,buy_date,sell_date,ticker,adjclose,stochastic_oscillator
0,2024-01-26,2024-01-29,2024-02-02,CHPT,1.94,2.8918
1,2024-01-26,2024-01-29,2024-02-02,AMC,4.07,2.3514
2,2024-01-26,2024-01-29,2024-02-02,AGL,5.82,2.4055
3,2024-01-26,2024-01-29,2024-02-02,PLUG,3.40,1.8559
4,2024-01-26,2024-01-29,2024-02-02,LCID,2.65,1.3774


In [10]:
db.connect()
db.drop("recommendations")
db.store("recommendations",recommendations)
db.disconnect()